# 04 - Embedding Generation

Generate vector embeddings for Forge knowledge chunks using an embedding model. These embeddings will later be stored in a vector database for semantic retrieval.

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from pathlib import Path
import json
import time
import numpy as np
import pandas as pd


from google.colab import drive

from sentence_transformers import SentenceTransformer

In [ ]:
drive.mount("/content/drive")

Mounted at /content/drive


In [10]:
with open(CONFIG["chunks_file"], "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(len(chunks))

4778


In [11]:
PROJECT_ROOT = Path("/content/drive/MyDrive/forge")

CONFIG = {
    "project_root": PROJECT_ROOT,
    "chunks_file": PROJECT_ROOT / "knowledge_base" / "chunks.json",
    "embeddings": PROJECT_ROOT / "knowledge_base" / "embeddings",
}

print(CONFIG["chunks_file"])

/content/drive/MyDrive/forge/knowledge_base/chunks.json


In [12]:
with open(CONFIG["chunks_file"], "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(len(chunks))

4778


In [13]:
MODEL_NAME = "BAAI/bge-base-en-v1.5"

model = SentenceTransformer(MODEL_NAME)

print("Embedding model loaded successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.


In [14]:
texts = [chunk["text"] for chunk in chunks]

print(f"Total texts: {len(texts)}")
print(f"Sample text length: {len(texts[0])}")

Total texts: 4778
Sample text length: 897


In [15]:
start_time = time.time()

embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

end_time = time.time()

print(f"Embedding generation completed.")
print(f"Time taken: {(end_time - start_time)/60:.2f} minutes")
print(f"Embedding shape: {embeddings.shape}")

Batches:   0%|          | 0/150 [00:00<?, ?it/s]

Embedding generation completed.
Time taken: 1.46 minutes
Embedding shape: (4778, 768)


In [16]:
embedding_file = CONFIG["project_root"] / "knowledge_base" / "embeddings.npy"

np.save(
    embedding_file,
    embeddings
)

print(f"Saved embeddings: {embedding_file}")
print(f"Shape: {embeddings.shape}")

Saved embeddings: /content/drive/MyDrive/forge/knowledge_base/embeddings.npy
Shape: (4778, 768)


In [17]:
metadata = []

for chunk in chunks:
    metadata.append({
        "chunk_id": chunk["chunk_id"],
        "technology": chunk["technology"],
        "technology_id": chunk["technology_id"],
        "category": chunk["category"],
        "organization": chunk["organization"],
        "license": chunk["license"],
        "priority": chunk["priority"],
        "update_frequency": chunk["update_frequency"],
        "source": chunk["source"],
        "url": chunk["url"],
        "path": chunk["path"],
        "chunk_index": chunk["chunk_index"],
        "text": chunk["text"],
        "character_count": chunk["character_count"]
    })

metadata_file = CONFIG["project_root"] / "knowledge_base" / "embedding_metadata.json"

with open(metadata_file, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f"Saved metadata: {metadata_file}")
print(f"Metadata entries: {len(metadata)}")

Saved metadata: /content/drive/MyDrive/forge/knowledge_base/embedding_metadata.json
Metadata entries: 4778


In [18]:
loaded_embeddings = np.load(
    CONFIG["project_root"] / "knowledge_base" / "embeddings.npy"
)

with open(
    CONFIG["project_root"] / "knowledge_base" / "embedding_metadata.json",
    "r",
    encoding="utf-8"
) as f:
    loaded_metadata = json.load(f)

print("Embedding shape:", loaded_embeddings.shape)
print("Metadata count :", len(loaded_metadata))

print("\nFirst metadata entry:")
print(json.dumps(loaded_metadata[0], indent=2, ensure_ascii=False))

Embedding shape: (4778, 768)
Metadata count : 4778

First metadata entry:
{
  "chunk_id": "9660c5d5-5b17-49be-a42f-1d01e3bee92e",
  "technology": "AWS SageMaker",
  "technology_id": "source-aws-sagemaker",
  "category": "deployment",
  "organization": "Amazon Web Services",
  "license": "",
  "priority": "high",
  "update_frequency": "weekly",
  "source": "official_documentation",
  "url": "https://docs.aws.amazon.com/sagemaker/",
  "path": "aws_sagemaker/official_documentation.json",
  "chunk_index": 0,
  "text": "Use Machine Learning Environments Learn about machine learning environments that Amazon SageMaker AI offers.\nLabel Data with a Human-in-the-loop Learn how to use a human-in-the-loop to help label data more accurately.\nCreate, Store, and Share Features Learn how to create, store, and share extracted data signals (features) for machine learning.\nUse Docker Containers to Build Models Learn how to use Docker containers to build your machine learning models.\nDetect Bias and U